In [11]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_path = project_root / "data" / "raw" / "Dataset Capstone.xlsx"
processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(
    raw_path,
    sheet_name="customer_survey"
)

df_clean = df_raw.copy()

print("Jumlah record:", len(df_clean))
display(df_clean["data_type"].value_counts(dropna=False))

Jumlah record: 100


data_type
dummy     98
actual     2
Name: count, dtype: int64

In [3]:
print("=== STANDARDISASI TEKS ===")

text_columns = [
    "response_id",
    "respondent_type",
    "information_needed",
    "main_information",
    "preferred_platform",
    "website_expectation",
    "additional_feedback",
    "consent",
    "data_type"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
    )

print("Standardisasi teks selesai.")

=== STANDARDISASI TEKS ===
Standardisasi teks selesai.


In [12]:
df_clean["data_type"] = (
    df_clean["data_type"]
    .str.lower()
)

allowed_types = ["actual", "dummy"]

assert df_clean["data_type"].isin(allowed_types).all(), \
    "Terdapat label data tidak valid!"

display(df_clean["data_type"].value_counts())

data_type
dummy     98
actual     2
Name: count, dtype: int64

In [13]:
df_clean["consent_bool"] = (
    df_clean["consent"]
    .str.lower()
    .map({
        "setuju": True,
        "tidak setuju": False
    })
    .astype("boolean")
)

print("Consent tidak dikenali:",
      df_clean["consent_bool"].isna().sum())

display(
    df_clean[
        ["response_id", "data_type", "consent", "consent_bool"]
    ].head()
)

Consent tidak dikenali: 0


,response_id,data_type,consent,consent_bool
0,R001,actual,Setuju,True
1,R002,actual,Setuju,True
2,R003,dummy,Setuju,True
3,R004,dummy,Setuju,True
4,R005,dummy,Setuju,True


In [14]:
df_clean["information_difficulty"] = pd.to_numeric(
    df_clean["information_difficulty"],
    errors="coerce"
)

invalid_difficulty = df_clean[
    ~df_clean["information_difficulty"].between(1, 5)
]

print("Jumlah skala tidak valid:", len(invalid_difficulty))
display(invalid_difficulty)

Jumlah skala tidak valid: 0


,response_id,submitted_at,respondent_type,information_needed,main_information,preferred_platform,information_difficulty,website_expectation,additional_feedback,consent,data_type,consent_bool


In [15]:
print("=== FORMAT INFORMATION NEEDED ===")

display(
    df_clean[
        ["response_id", "information_needed"]
    ].head(10)
)

print("\nContoh kombinasi jawaban:")

display(
    df_clean["information_needed"]
    .value_counts(dropna=False)
    .head(15)
)

=== FORMAT INFORMATION NEEDED ===


,response_id,information_needed
0,R001,"Jam Operasional, Lokasi, Fasilitas, Harga, Menu"
1,R002,"Promo/Event, Jam Operasional"
2,R003,"Promo/Event, Lokasi, Jam Operasional"
3,R004,"Harga, Fasilitas, Lokasi"
4,R005,"Promo/Event, Menu, Lokasi, Jam Operasional, La..."
5,R006,"Harga, Menu"
6,R007,"Fasilitas, Menu, Promo/Event, Lokasi, Harga"
7,R008,"Menu, Harga, Promo/Event"
8,R009,"Promo/Event, Jam Operasional, Lokasi, Menu"
9,R010,"Fasilitas, Harga, Lokasi, Jam Operasional"



Contoh kombinasi jawaban:


information_needed
Promo/Event, Jam Operasional                            2
Harga, Menu                                             2
Jam Operasional, Menu, Promo/Event, Harga, Fasilitas    2
Fasilitas, Lokasi, Jam Operasional                      2
Harga, Lokasi                                           2
Jam Operasional, Lokasi, Harga, Fasilitas               2
Menu, Harga, Lokasi, Jam Operasional                    2
Jam Operasional, Lokasi, Fasilitas, Harga, Menu         1
Promo/Event, Lokasi, Jam Operasional                    1
Harga, Fasilitas, Lokasi                                1
Promo/Event, Menu, Lokasi, Jam Operasional, Lainnya     1
Fasilitas, Menu, Promo/Event, Lokasi, Harga             1
Menu, Harga, Promo/Event                                1
Promo/Event, Jam Operasional, Lokasi, Menu              1
Fasilitas, Harga, Lokasi, Jam Operasional               1
Name: count, dtype: int64

In [16]:
print("=== PEMISAHAN DATA SURVEI ===")

df_actual = df_clean.loc[
    (df_clean["data_type"] == "actual") &
    (df_clean["consent_bool"] == True)
].copy()

df_dummy = df_clean.loc[
    df_clean["data_type"] == "dummy"
].copy()

print("Jumlah actual:", len(df_actual))
print("Jumlah dummy:", len(df_dummy))
print("Total terpisah:", len(df_actual) + len(df_dummy))

=== PEMISAHAN DATA SURVEI ===
Jumlah actual: 2
Jumlah dummy: 98
Total terpisah: 100


In [17]:
print("=== VALIDASI AKHIR SURVEI ===")

assert len(df_clean) == len(df_raw), \
    "Jumlah record berubah!"

assert df_clean["response_id"].is_unique, \
    "Terdapat response_id duplikat!"

assert df_clean["data_type"].isin(
    ["actual", "dummy"]
).all(), "Jenis data tidak valid!"

assert df_clean["information_difficulty"].between(
    1, 5
).all(), "Skala kesulitan tidak valid!"

assert set(df_actual["response_id"]).isdisjoint(
    set(df_dummy["response_id"])
), "Data actual dan dummy tumpang tindih!"

eligible_actual = (
    (df_clean["data_type"] == "actual") &
    (df_clean["consent_bool"] == True)
).sum()

assert len(df_actual) == eligible_actual

assert len(df_dummy) == (
    df_clean["data_type"] == "dummy"
).sum()

print("Jumlah record awal:", len(df_raw))
print("Jumlah record bersih:", len(df_clean))
print("Actual layak analisis:", len(df_actual))
print("Dummy:", len(df_dummy))

print("\nValidasi dasar berhasil!")

=== VALIDASI AKHIR SURVEI ===
Jumlah record awal: 100
Jumlah record bersih: 100
Actual layak analisis: 2
Dummy: 98

Validasi dasar berhasil!


In [18]:
df_clean.to_csv(
    processed_dir / "survey_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_actual.to_csv(
    processed_dir / "survey_actual.csv",
    index=False,
    encoding="utf-8-sig"
)

df_dummy.to_csv(
    processed_dir / "survey_dummy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tiga dataset berhasil disimpan:")
print("survey_clean.csv")
print("survey_actual.csv")
print("survey_dummy.csv")

Tiga dataset berhasil disimpan:
survey_clean.csv
survey_actual.csv
survey_dummy.csv


In [19]:
cleaning_log = pd.DataFrame([
    {
        "step": 1,
        "process": "Standardisasi teks",
        "description": "Menghapus spasi awal dan akhir pada kolom teks"
    },
    {
        "step": 2,
        "process": "Standardisasi data_type",
        "description": "Menyeragamkan penulisan label actual dan dummy"
    },
    {
        "step": 3,
        "process": "Standardisasi consent",
        "description": "Membuat consent_bool dari jawaban consent"
    },
    {
        "step": 4,
        "process": "Validasi skala",
        "description": "Memeriksa information_difficulty pada rentang 1-5"
    },
    {
        "step": 5,
        "process": "Pemisahan dataset",
        "description": "Memisahkan respons actual dan dummy"
    },
    {
        "step": 6,
        "process": "Penanganan missing values",
        "description": "Mempertahankan additional_feedback kosong karena opsional"
    },
    {
        "step": 7,
        "process": "Koreksi metadata",
        "description": (
            "Menggunakan label asal terbaru: "
            "2 actual dan 98 dummy, berdasarkan catatan pengumpulan"
        )
    }
])

cleaning_log.to_csv(
    reports_dir / "survey_cleaning_log.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cleaning_log)

print("Cleaning log berhasil disimpan.")

,step,process,description
0,1,Standardisasi teks,Menghapus spasi awal dan akhir pada kolom teks
1,2,Standardisasi data_type,Menyeragamkan penulisan label actual dan dummy
2,3,Standardisasi consent,Membuat consent_bool dari jawaban consent
3,4,Validasi skala,Memeriksa information_difficulty pada rentang 1-5
4,5,Pemisahan dataset,Memisahkan respons actual dan dummy
5,6,Penanganan missing values,Mempertahankan additional_feedback kosong kare...
6,7,Koreksi metadata,Menggunakan label asal terbaru: 2 actual dan 9...


Cleaning log berhasil disimpan.
